# Lab 6 — Linear Regression Evaluation Metrics

**Day 02 · Python for Data Science · Cisco AI/ML Training**

---

## Learning objectives

1. Split data into **train** and **test** sets with `train_test_split`.
2. Fit regression on **training** data only; evaluate on **held-out** test data.
3. Compute and interpret **R²**, **MSE**, **MAE**, and **RMSE**.
4. Relate metric magnitude to the business scale of restaurant ratings.

> **Checkpoints:** train **400** / test **100** · RMSE ≈ **0.69** · R² ≈ **-0.003**

**Companion script:** `../scripts/lab06_lr_evaluation_metrics.py`


## Why train/test split?

| Approach | Risk |
|----------|------|
| Train and evaluate on **same** data | Overly optimistic scores (memorization) |
| **Hold-out** test set | Simulates performance on unseen restaurants |

We use `random_state=42` for **reproducible** splits across the class.

**Golden rule:** Never tune or pick features using test-set performance directly.


## Metric cheat sheet

| Metric | Formula idea | Better when… |
|--------|--------------|--------------|
| **R²** | Fraction of variance explained | Closer to **1** |
| **MSE** | Mean squared error | Closer to **0** |
| **MAE** | Mean absolute error | Closer to **0** (same units as rating) |
| **RMSE** | √MSE | Closer to **0**; penalizes large errors more than MAE |

**R² can be negative** when the model is worse than predicting the mean — common with weak features on synthetic data.


---

## 1. Load data (same features as Lab 5)


In [ ]:
%matplotlib inline

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

GH_ROOT = Path.cwd().resolve()
if GH_ROOT.name == "notebooks":
    GH_ROOT = GH_ROOT.parents[2]
elif GH_ROOT.name == "day-02":
    GH_ROOT = GH_ROOT.parents[1]
else:
    for parent in [GH_ROOT, *GH_ROOT.parents]:
        if (parent / "data" / "zomato" / "zomato_restaurants.csv").is_file():
            GH_ROOT = parent
            break

df = pd.read_csv(GH_ROOT / "data" / "zomato" / "zomato_restaurants.csv")
X = df[["votes", "average_cost_for_two"]]
y = df["aggregate_rating"]
print(f"Full dataset: {X.shape[0]} rows")


---

## 2. Train/test split


In [ ]:
TEST_SIZE = 0.2
RANDOM_STATE = 42

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

print(f"train size: {len(X_train)}")
print(f"test size:  {len(X_test)}")

assert len(X_train) == 400 and len(X_test) == 100
print("✓ Split checkpoint OK")


---

## 3. Fit on training data only


In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f"intercept: {model.intercept_:.4f}")
print(f"coefficients: {model.coef_.round(6)}")


---

## 4. Compute all four metrics


In [ ]:
r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = mse ** 0.5

metrics = pd.DataFrame({
    "metric": ["R2", "MSE", "MAE", "RMSE"],
    "value": [r2, mse, mae, rmse],
})
display(metrics.round(4))

print("Lab 6 — LR evaluation metrics")
print(f"R2: {r2:.4f}")
print(f"MSE: {mse:.4f}")
print(f"MAE: {mae:.4f}")
print(f"RMSE: {rmse:.4f}")


### Interpreting RMSE on ratings

Ratings span roughly **2.5 – 4.9**. An RMSE of ~**0.69** means typical errors are under one star — but R² near zero says the model barely beats guessing the average.

**Discussion:** Would you deploy this model? What features might improve it?


---

## 5. Visual diagnostics

### Predicted vs actual (test set)


In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
sns.scatterplot(x=y_test, y=y_pred, ax=ax, alpha=0.6)
ax.plot([y.min(), y.max()], [y.min(), y.max()], "r--", label="perfect prediction")
ax.set_xlabel("actual rating")
ax.set_ylabel("predicted rating")
ax.set_title("Test set: predicted vs actual")
ax.legend()
plt.tight_layout()
plt.show()


Points on the red dashed line = perfect predictions. Wide scatter → weak model.


In [ ]:
residuals = y_test - y_pred
fig, ax = plt.subplots(figsize=(6, 4))
sns.histplot(residuals, bins=15, kde=True, ax=ax)
ax.set_title("Test residuals")
ax.set_xlabel("actual - predicted")
plt.tight_layout()
plt.show()


---

## 6. Experiment — change `test_size`

Re-run the split cell with `TEST_SIZE = 0.3` and observe new train/test sizes and metrics.


In [ ]:
# Optional experiment (uncomment to run):
# X_train2, X_test2, y_train2, y_test2 = train_test_split(X, y, test_size=0.3, random_state=42)
# m2 = LinearRegression().fit(X_train2, y_train2)
# p2 = m2.predict(X_test2)
# print("30% test RMSE:", (mean_squared_error(y_test2, p2) ** 0.5).round(4))

print("Default: test_size=0.2, train=400, test=100")


---

## 7. Compare to naive baseline

A **baseline** that always predicts the training mean:


In [ ]:
baseline_pred = [y_train.mean()] * len(y_test)
r2_baseline = r2_score(y_test, baseline_pred)
rmse_baseline = mean_squared_error(y_test, baseline_pred) ** 0.5

print(f"Baseline (mean) R2: {r2_baseline:.4f}")
print(f"Baseline RMSE:      {rmse_baseline:.4f}")
print(f"Our model R2:       {r2:.4f}")
print(f"Our model RMSE:     {rmse:.4f}")


If our model R² ≤ 0, it does not beat the mean baseline — time to engineer better features or try non-linear models (later days).


---

## 8. Final checkpoint


In [ ]:
assert len(X_train) == 400 and len(X_test) == 100
assert abs(rmse - 0.6852) < 0.02
print("✓ All checkpoint assertions passed")
print("\nDay 02 complete — Day 03 introduces classification with Lending Club.")


## Concept — L1 / L2 regularization

<!-- cisco-enrich-2026-06 -->

Plain linear regression can **overfit** with many correlated features. **Ridge (L2)** shrinks coefficients; **Lasso (L1)** can zero them out for feature selection.


## Extension — Ridge and Lasso on Zomato

In [ ]:
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
import numpy as np

X = df[["votes", "average_cost_for_two"]]
y = df["aggregate_rating"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

rows = []
for name, model in [
    ("OLS", LinearRegression()),
    ("Ridge", Ridge(alpha=1.0)),
    ("Lasso", Lasso(alpha=0.01, max_iter=5000)),
]:
    model.fit(X_tr, y_tr)
    pred = model.predict(X_te)
    rows.append({"model": name, "rmse": float(np.sqrt(mean_squared_error(y_te, pred)))})
print(pd.DataFrame(rows).round(4))


---

## Reflection questions

1. Why must we call `.fit()` only on `X_train`, not the full dataset?
2. Which metric would you report to a business stakeholder — MAE or RMSE? Why?
3. How does this lab connect to **model evaluation** in the CRISP-DM cycle (Day 1)?

**Previous:** [Lab 5 — Linear regression fit](lab05_linear_regression_fit.ipynb)  
**Next:** Day 03 — Probability & logistic regression (Lending Club)
